# Import libraries

In [109]:
import pandas as pd
import numpy as np
import copy

import gc
from pathlib import Path

import os
from brat_parser import get_entities_relations_attributes_groups
from itertools import combinations

In [128]:
import torch
from torch import nn

from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, AutoConfig
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, roc_auc_score

In [111]:
os.environ["HF_HUB_OFFLINE"] = "1"

# Config

In [112]:
class config:
    data_folder = "../data"
    model_directory = '/home/shared/Models/'
    model_names = ['BioClinical-ModernBERT-base','Bio_ClinicalBERT',' Clinical_ModernBERT']

# Helper Functions

In [113]:
def get_file_names(directory):
    file_names = []
    # List all files in the directory
    for filename in os.listdir(directory):
        # Check if the current entry is a file
        if os.path.isfile(os.path.join(directory, filename)):
            file_names.append(os.path.join(directory, filename))
    return file_names

In [114]:
def get_relationships(input_string):
  lines = input_string.split('\n')
  e_lines = [line for line in lines if line.startswith('E')]
  result = []
  for line in e_lines:
      pairs = line.split()
      buffer=[]
      buffer2={}
      for t in pairs[1:]:
        buffer.append(t.split(':')[1])
      for buff in buffer[1:]:
        #buffer2.append((buffer[0],buff))
        buffer2[buff]=buffer[0]
      result.append(buffer2)
  return result

def get_sdoh_data_dict(txt_files,ann_files):
  data_list=[]
  for txt,ann in zip(txt_files,ann_files):
    file=open(txt,'r')
    data_dict={}

#

    data_dict['clinical_note']=file.read()
    data_dict['filename']= re.search(r"\/mimic\/(.*?)\.txt",txt).groups()[0]
    file.close()
    entities, relations, attributes, groups = get_entities_relations_attributes_groups(ann)
    data_dict['entities']=entities
    data_dict['relations']=get_relationships(open(ann,'r').read())
    data_dict['attributes']=attributes
    data_dict['groups']=groups
    data_list.append(data_dict)
    new_dict = {}
    for i, dict_ in enumerate(data_dict['relations']):


      for k, v in dict_.items():
          if v in new_dict:
              new_dict[v].append(k)
          else:
              new_dict[v] = [k]
    data_dict['relations']=new_dict

      #####################################################
    event_type_dict={'Alcohol':'ALC',  'Drug':'DRG',  'Employment':'EMP',
        'LivingStatus':'LVS', 'Tobacco':'TOB'}


    new_relations=[]
    id_enti={}
    for enti in data_dict['entities'].values():
      if enti.type in list(event_type_dict.keys()):
        id_enti[enti.id]=enti.type

    result = [(id_enti[key], [key]+data_dict['relations'][key]) for key in id_enti]
    data_dict['event_rel']=result
    data_dict['enti_id_ignore']=list(id_enti.keys())

  return data_list

In [115]:
def get_target_clincal(sample):
    clinical_text = sample['clinical_note']

    # Initialize the label dictionary for the multi-class multi-label output
    label = {
        'Alcohol': None,
        'Tobacco': None,
        'Drug': None,
        'LivingStatus': None,
        'Employment': None
    }

    # A helper function to find the value for a category in the attributes
    def get_attribute_value(category, target_ids, attribute_type):
        for attr_key, attribute in sample['attributes'].items():
            if attribute.target in target_ids and attribute.type == attribute_type:
                return attribute.values[0]  # Assuming only one value per attribute
        return None

    # Process each category in the event_rel
    for category, target_ids in sample['event_rel']:
        if category == 'Tobacco':
            # Check if Tobacco is in the event_rel and get its attribute value
            label['Tobacco'] = get_attribute_value('Tobacco', target_ids, 'StatusTimeVal')
        elif category == 'Alcohol':
            # Check if Alcohol is in the event_rel and get its attribute value
            label['Alcohol'] = get_attribute_value('Alcohol', target_ids, 'StatusTimeVal')
        elif category == 'Drug':
            # Check if Drug is in the event_rel and get its attribute value
            label['Drug'] = get_attribute_value('Drug', target_ids, 'StatusTimeVal')
        elif category == 'LivingStatus':
            # LivingStatus can have two different types, StatusTimeVal and TypeLivingVal
            # First check for StatusTimeVal (current, past, future)
            status_value = get_attribute_value('LivingStatus', target_ids, 'StatusTimeVal')
            # Then check for TypeLivingVal (alone, with family, etc.)
            type_value = get_attribute_value('LivingStatus', target_ids, 'TypeLivingVal')

            if status_value and type_value:
                # Multiply the values by concatenating them as a string (you can change the format here if needed)
                label['LivingStatus'] = f'{status_value}_{type_value}'
            else:
                # If one or both are missing, return None
                label['LivingStatus'] = None
        elif category == 'Employment':
            # Employment has values like employed, unemployed, retired, etc.
            label['Employment'] = get_attribute_value('Employment', target_ids, 'StatusEmployVal')


    # Define all possible values for each label
    alcohol_values = ['none', 'current', 'past']
    tobacco_values = ['none', 'current', 'past']
    drug_values = ['none', 'current', 'past']
    living_status_values = [
        'current_with_family', 'current_alone', 'current_with_others', 'current_homeless',
        'past_with_family', 'past_alone', 'past_with_others', 'past_homeless',
        'future_with_family', 'future_alone', 'future_with_others', 'future_homeless'
    ]
    employment_values = ['employed', 'unemployed', 'retired', 'on_disability', 'student', 'homemaker']


    # Function to return the index of the value in the list or -1 if None
    def get_value_index(value, value_list):
        if value is None:
            return 0  
        else:
            try:
                return value_list.index(value)+1  # Return the index of the value
            except ValueError:
                return 0  # In case the value is not found (shouldn't happen if data is correct)

    # Getting the numerical representation (index) of the labels
    numerical_labels = {
        'Alcohol': get_value_index(label['Alcohol'], alcohol_values),
        'Tobacco': get_value_index(label['Tobacco'], tobacco_values),
        'Drug': get_value_index(label['Drug'], drug_values),
        # 'LivingStatus': get_value_index(label['LivingStatus'], living_status_values),
        'Employment': get_value_index(label['Employment'], employment_values)
    }

    # Return the clinical_text along with the labels
    return clinical_text, numerical_labels

In [116]:
# compute_metrics function for evaluation
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    print(logits.shape)
    logits = torch.sigmoid(torch.tensor(logits))
    predictions = (logits > 0.5).detach().cpu().numpy()

    labels = np.asarray(labels).astype(int)


    results = {}
    results['accuracy'] = (predictions == labels).mean().item()

    f1_scores = f1_score(labels, predictions, average=None)

    # Create a dictionary of F1-scores for each class
    f1_score_dict = {f'class_{i}': f1_scores[i] for i in range(len(f1_scores))}

    overall_f1_score = f1_score(labels, predictions, average='macro')
    results['overall_f1'] = overall_f1_score

    try:
        roc_auc = roc_auc_score(labels, logits, average='macro')
    except ValueError:
        roc_auc = float('nan')  # Handle cases where ROC AUC cannot be computed (e.g., no positive cases)

    results['roc_auc'] = roc_auc
    return {**results, **f1_score_dict}


# Data

In [117]:
sdoh_path_train= config.data_folder + '/SHAC_all_mimic/train/mimic'
sdoh_path_dev= config.data_folder + '/SHAC_all_mimic/dev/mimic'
sdoh_path_test= config.data_folder + '/SHAC_all_mimic/test/mimic'

# file_list = get_file_names(directory_path)
umass_train_file_list=get_file_names(sdoh_path_train)
umass_dev_file_list=get_file_names(sdoh_path_dev)
umass_test_file_list=get_file_names(sdoh_path_test)

In [118]:
config.data_folder + '/SHAC_all_mimic/train/mimic'

'../data/SHAC_all_mimic/train/mimic'

In [119]:
print("Number of files in train_data: ", len(umass_train_file_list))
print("Number of files in dev_data: ", len(umass_dev_file_list))
print("Number of files in test_data: ", len(umass_test_file_list))

Number of files in train_data:  2632
Number of files in dev_data:  376
Number of files in test_data:  746


In [120]:
import re

regex_for_txt_umass = r'(?<=\.\./data/SHAC_all_mimic/train/mimic/)\d.*?\.txt$'
regex_for_ann_umass = r'(?<=\.\./data/SHAC_all_mimic/train/mimic/)\d.*?\.ann$'

regex_for_txt_umass_dev = r'(?<=\.\./data/SHAC_all_mimic/dev/mimic/)\d.*?\.txt$'
regex_for_ann_umass_dev = r'(?<=\.\./data/SHAC_all_mimic/dev/mimic/)\d.*?\.ann$'

regex_for_txt_umass_test = r'(?<=\.\./data/SHAC_all_mimic/test/mimic/)\d.*?\.txt$'
regex_for_ann_umass_test = r'(?<=\.\./data/SHAC_all_mimic/test/mimic/)\d.*?\.ann$'

txt_files_umass_train=[]
ann_files_umass_train=[]

txt_files_umass_dev=[]
ann_files_umass_dev=[]

txt_files_umass_test=[]
ann_files_umass_test=[]


for dir in umass_train_file_list:
  if re.search(regex_for_txt_umass,dir):
    txt_files_umass_train.append(dir)
  if re.search(regex_for_ann_umass,dir):
    ann_files_umass_train.append(dir)

for dir in umass_dev_file_list:
  if re.search(regex_for_txt_umass_dev,dir):
    txt_files_umass_dev.append(dir)
  if re.search(regex_for_ann_umass_dev,dir):
    ann_files_umass_dev.append(dir)

for dir in umass_test_file_list:
  if re.search(regex_for_txt_umass_test,dir):
    txt_files_umass_test.append(dir)
  if re.search(regex_for_ann_umass_test,dir):
    ann_files_umass_test.append(dir)


In [121]:
txt_files_umass_train.sort()
ann_files_umass_train.sort()
txt_files_umass_dev.sort()
ann_files_umass_dev.sort()
txt_files_umass_test.sort()
ann_files_umass_test.sort()

In [122]:
umass_data_train=get_sdoh_data_dict(txt_files_umass_train,ann_files_umass_train)
umass_data_dev=get_sdoh_data_dict(txt_files_umass_dev,ann_files_umass_dev)
umass_data_test=get_sdoh_data_dict(txt_files_umass_test,ann_files_umass_test)

In [123]:
umass_data_train=[data for data in umass_data_train if data['entities']!={}]
umass_data_dev=[data for data in umass_data_dev if data['entities']!={}]
umass_data_test=[data for data in umass_data_test if data['entities']!={}]

In [124]:
clinical_notes_train =[]
clinical_notes_dev =[]
clinical_notes_test =[]

labels_train =[]
labels_dev =[]
labels_test =[]

for sample in umass_data_train:
    clinical_note,label = get_target_clincal(sample)
    clinical_notes_train.append(clinical_note)
    labels_train.append(label)

for sample in umass_data_dev:
    clinical_note,label = get_target_clincal(sample)
    clinical_notes_dev.append(clinical_note)
    labels_dev.append(label)

for sample in umass_data_test:
    clinical_note,label = get_target_clincal(sample)
    clinical_notes_test.append(clinical_note)
    labels_test.append(label)

In [125]:
np.unique([val['Employment'] for val in labels_train])

array([0, 1, 2, 3, 4, 5, 6])

# Model

In [126]:
from transformers.modeling_outputs import SequenceClassifierOutput
class MultiHeadSequenceClassifier(nn.Module):
    def __init__(
        self,
        model_dir,
        head_sizes,
        local_files_only=True,
        dropout=None,
    ):
        super().__init__()

        self.head_sizes = dict(head_sizes)
        self.head_names = list(self.head_sizes.keys())

        # Load configuration and backbone only—not a pre-existing
        # AutoModelForSequenceClassification classification head.
        self.config = AutoConfig.from_pretrained(
            model_dir,
            local_files_only=local_files_only,
        )

        self.encoder = AutoModel.from_pretrained(
            model_dir,
            config=self.config,
            local_files_only=local_files_only,
        )

        hidden_size = self.config.hidden_size

        if dropout is None:
            dropout = getattr(self.config, "classifier_dropout", None)

        if dropout is None:
            dropout = getattr(self.config, "hidden_dropout_prob", 0.1)

        self.dropout = nn.Dropout(dropout)

        self.heads = nn.ModuleDict({
            head_name: nn.Linear(hidden_size, n_classes)
            for head_name, n_classes in self.head_sizes.items()
        })

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        **kwargs,
    ):

        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            **kwargs,
        )

        pooled_output = encoder_outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)

        logits_per_head = [
            self.heads[head_name](pooled_output)
            for head_name in self.head_names
        ]

        # Concatenated tensor is Trainer-friendly:
        # [batch, total number of classes across all heads]
        logits = torch.cat(logits_per_head, dim=-1)

        return SequenceClassifierOutput(
            loss=None,
            logits=logits,
        )

# Dataset and Data Loaders

In [134]:
class ClinicalNotesDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512, label_columns=None):
        assert len(texts) == len(labels), (
            f"Length mismatch: {len(texts)} texts vs {len(labels)} labels"
        )

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Fix a stable target order for multi-label learning
        self.label_columns = (
            label_columns if label_columns is not None
            else list(labels[0].keys())
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label_dict = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        labels = torch.tensor(
            [label_dict[col] for col in self.label_columns],
            dtype=torch.float32,
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }

        
HEAD_SIZES = {
        "alcohol": 4,
        "tobacco": 4,
        "drug": 4,
        "employement":7
    }

# Equal contribution of each target to final training loss.
# Change these only if certain targets should carry more importance.
TASK_WEIGHTS = {
        "alcohol": 1.0,
        "tobacco": 1.0,
        "drug": 1.0,
        "employement":1.0
    }

HEAD_NAMES = list(HEAD_SIZES.keys())
N_HEADS = len(HEAD_SIZES)
TOTAL_LOGITS = sum(HEAD_SIZES.values())


# Load the tokenizer and model
model_dir = config.model_directory + config.model_names[0]

tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
model = MultiHeadSequenceClassifier(
    model_dir=model_dir,
    head_sizes={
        "alcohol": 4,
        "tobacco": 4,
        "drug": 4,
        "employement":7
    },
    local_files_only=True,
)

# Prepare datasets
train_dataset = ClinicalNotesDataset(clinical_notes_train, labels_train, tokenizer)
dev_dataset = ClinicalNotesDataset(clinical_notes_dev, labels_dev, tokenizer)
test_dataset = ClinicalNotesDataset(clinical_notes_test, labels_test, tokenizer)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 134/134 [00:00<00:00, 4832.60it/s]
[transformers] ModernBertModel LOAD REPORT from: /home/shared/Models/BioClinical-ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [140]:
train_dataset.labels[2]

{'Alcohol': 2, 'Tobacco': 2, 'Drug': 1, 'Employment': 2}

# Training

In [135]:
# Freeze pretrained Transformer backbone.
for param in model.encoder.parameters():
    param.requires_grad = False

# Train only the custom multiclass heads.
for param in model.heads.parameters():
    param.requires_grad = True


# Verify trainable versus frozen parameters.
trainable_params = 0
frozen_params = 0
total_params = 0

for name, param in model.named_parameters():
    n_params = param.numel()
    total_params += n_params

    if param.requires_grad:
        trainable_params += n_params
        print(f"TRAINABLE: {name:70s} | {n_params:,}")
    else:
        frozen_params += n_params

print("\nParameter summary")
print(f"Trainable: {trainable_params:,} / {total_params:,} "
      f"({100 * trainable_params / total_params:.4f}%)")
print(f"Frozen:    {frozen_params:,} / {total_params:,} "
      f"({100 * frozen_params / total_params:.4f}%)")

TRAINABLE: heads.alcohol.weight                                                   | 3,072
TRAINABLE: heads.alcohol.bias                                                     | 4
TRAINABLE: heads.tobacco.weight                                                   | 3,072
TRAINABLE: heads.tobacco.bias                                                     | 4
TRAINABLE: heads.drug.weight                                                      | 3,072
TRAINABLE: heads.drug.bias                                                        | 4
TRAINABLE: heads.employement.weight                                               | 5,376
TRAINABLE: heads.employement.bias                                                 | 7

Parameter summary
Trainable: 14,611 / 149,028,883 (0.0098%)
Frozen:    149,014,272 / 149,028,883 (99.9902%)


In [136]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    # logging_dir='./logs',
    logging_steps=10,
)

class MultiHeadTrainer(Trainer):
    """
    Computes weighted mean of CrossEntropyLoss values across heads.

    labels:
        Tensor of shape [batch_size, n_heads]
        dtype torch.long

    logits:
        Tensor of shape [batch_size, TOTAL_LOGITS]
    """

    def __init__(
        self,
        *args,
        head_sizes,
        task_weights=None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.head_sizes = dict(head_sizes)
        self.head_names = list(self.head_sizes.keys())

        if task_weights is None:
            task_weights = {
                head_name: 1.0
                for head_name in self.head_names
            }

        if set(task_weights.keys()) != set(self.head_names):
            raise ValueError(
                "task_weights must contain exactly the same keys as head_sizes."
            )

        self.task_weights = task_weights

        # Required if num_items_in_batch is accepted but deliberately unused.
        self.model_accepts_loss_kwargs = False

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        # Do not mutate the original batch dictionary.
        labels = inputs["labels"].long()
        model_inputs = {
            key: value
            for key, value in inputs.items()
            if key != "labels"
        }

        outputs = model(**model_inputs)
        logits = outputs.logits

        head_losses = []
        start_idx = 0

        for head_idx, (head_name, n_classes) in enumerate(
            self.head_sizes.items()
        ):
            end_idx = start_idx + n_classes

            # [batch, n_classes]
            head_logits = logits[:, start_idx:end_idx]

            # [batch], class IDs in [0, n_classes - 1]
            head_labels = labels[:, head_idx]

            head_loss = nn.functional.cross_entropy(
                head_logits,
                head_labels,
                # If target is sometimes missing, use -100 in the dataset:
                # ignore_index=-100,
            )

            weighted_head_loss = (
                self.task_weights[head_name] * head_loss
            )
            head_losses.append(weighted_head_loss)

            start_idx = end_idx

        total_weight = sum(self.task_weights.values())
        loss = torch.stack(head_losses).sum() / total_weight

        return (loss, outputs) if return_outputs else loss


In [137]:
trainer = MultiHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    head_sizes=HEAD_SIZES,
    task_weights=TASK_WEIGHTS,
)

trainer.train()

eval_results = trainer.evaluate()
print(eval_results)

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
